# This notebook generates linearly evolved (non-)Gaussian primordial fields

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm
import camb

from DensityFields import DensityField2D

In [ ]:
#zi = 0 #Index of desired redshift 
zi = np.array([0,3,10,30,50,100])
BoxSize = 1000. #Size of the periodic box in Mpc/h
grid = 128 #Size of the grid
cell_size = BoxSize/grid #Physical size of a cell
kF = 2*np.pi / BoxSize #Fundamental mode of the box
kNyq = grid/2 * kF #Nyquist frequency of the box

# Cosmology settings
cosmo_params = {
    'h': 0.6711,
    'r': 0,
    'As': 2.13e-09,
    'A': 2*np.pi*np.pi*2.13e-09,
    'ns': 0.9624,
    'kpivot': 0.05,
    'z_recomb': 1090.48,
    'ombh2': 0.02233,
    'omch2': 0.1198,
    'tau': 0.0561,
    'lmax': 11000,
    'accuracy_boost': 4,
}

# simulations settings
num_sim = 100000 # number of files to generate
num_bispectra = 1000
fnl_range=(-1000, 1000)

data_dir='data/camb_3'
base_name = f'{grid}x{num_sim//1000}k_fnl{fnl_range[0]}-{fnl_range[1]}'
data_file = f"{base_name}.npy"
fnl_file = f"{base_name}-fnls.npy"

if not os.path.exists(data_dir): os.makedirs(data_dir)

## Plot the Transfer functions for test, compaire with CAMB Demo

In [ ]:
# Compare to Box 18 in https://camb.readthedocs.io/en/latest/CAMBdemo.html

# FFT_map = DensityField2D(BoxSize,grid,n_threads=1, cosmo_params=cosmo_params)
# transfer = FFT_map.results.get_cmb_transfer_data() # type: ignore
# fig, axs = plt.subplots(2,2, figsize=(12,8), sharex = True)
# for ix, ax in zip([3, 20, 40, 60],axs.reshape(-1)):
#     ax.plot(transfer.q, transfer.delta_p_l_k[0,ix,:])
#     ax.set_xlim(0,0.6)
#     ax.set_title(r'$\ell = %s$'%transfer.L[ix])
#     if ix>1: ax.set_xlabel(r'$k \rm{Mpc}$')
# fig.suptitle('CMB transfer function')
# plt.show()

In [ ]:
# Box 19
# def plot_cmb_transfer_l(trans, ix):
#     _, axs = plt.subplots(1,2, figsize=(12,6))
#     for source_ix, (name, ax) in enumerate(zip(['T', 'E'], axs)):
#         ax.semilogx(trans.q,trans.delta_p_l_k[source_ix,ix,:])
#         ax.set_xlim([1e-5, 0.05])
#         ax.set_xlabel(r'$k \rm{Mpc}$')
#         ax.set_title(r'%s transfer function for $\ell = %s$'%(name, trans.L[ix]))
# plot_cmb_transfer_l(transfer,0)

## The class DensityField2D will set up everything that we need for generating maps aswell as measuring powerspectra and bispectra

In [ ]:
# Instantiate a density field
FFT_map = DensityField2D(BoxSize,grid,n_threads=1, cosmo_params=cosmo_params)
 
# Generate a linear realization with fnl=1
FFT_map.GenerateCAMBField(fnl=1.,seed=0,verbose=True, debug_plots=True)

In [ ]:
ks, pks, ns = FFT_map.Pk()
plt.loglog(ks, pks)
plt.xlabel('k')
plt.ylabel('Power')
plt.title('1D Power Spectrum')
plt.show()

ells = np.round(ks * FFT_map.d_A * FFT_map.h)
cl = ells * (ells + 1) / ( 2 * np.pi) * pks
plt.semilogy(ells, cl)
plt.xlabel('$\ell$')
plt.ylabel('$D_\ell$')
plt.show()

In [ ]:
FFT_map = DensityField2D(BoxSize,grid,n_threads=1, cosmo_params=cosmo_params)

# Generate the same field but cut off at grid/3*kF (where FFT bispectrum measurements start to fail)
FFT_map.GenerateCAMBField(0,fnl=1.,k_cut_high=grid/3*kF,seed=None,verbose=False)
plt.imshow(FFT_map.r_delta)
plt.title('density field with k_cut_high {}'.format(grid/3*kF))
plt.show()

kk, Pk, _ = FFT_map.Pk()
plt.loglog(kk,Pk)
plt.ylabel("$P(k)$ [Mpc/h]$^3$")
plt.xlabel("$k$ [h/Mpc]")
plt.show()

## Computing Bispectra can be done as follows

In [ ]:
%%time
# Compute the bispectrum in a given binning:
BBB = FFT_map.Bk(2.5,3,13,'All',verbose=True)
plt.semilogy(BBB[:,-2])
plt.ylabel("$B(k)$ [Mpc/h]$^6$")
plt.xlabel("triangle$_i$")
print("kmax =",(BBB[-1,0]+1.5)*kF,grid/3*kF)

## Let's perform a Fisher forecast

In [ ]:
df_base = DensityField2D(BoxSize,grid, n_threads=1, cosmo_params=cosmo_params)

def bispectrum(fnl,seed, ls, qs, transfers, ells, d_A, cosmo, verbose=False):
    FFT_map = DensityField2D(BoxSize,grid, n_threads=1, d_A=d_A, ls=ls, qs=qs, transfers=transfers, ells=ells, cosmo_params=cosmo, verbose=verbose)
    FFT_map.GenerateCAMBField(k_cut_high=None,fnl=fnl,seed=seed,verbose=verbose)
    BBB = FFT_map.Bk(2.5,3,13,'All',verbose=verbose)
    return BBB

### We compute many bispectra with fixed amounts of pnG

In [10]:
%%time
BispecP = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(bispectrum)(100.,i, df_base.Ls, df_base.qs, df_base.transfer, df_base.ells, df_base.d_A, df_base.cosmo) for i in range(num_bispectra)]))

In [ ]:
%%time
BispecM = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(bispectrum)(-100.,i,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo) for i in range(num_bispectra)]))

In [ ]:
BispecG = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(bispectrum)(0.,i, df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo) for i in range(num_bispectra)]))

### We compute the covariance matrix and its inverse, corrected by the Hartlap factor

In [ ]:
Cov = np.cov(BispecG[:,:,-2].T)

hartlapfactor = (len(BispecG) - len(Cov) - 2) / (len(BispecG) - 1)
Cov_Inv = np.linalg.inv(Cov)
Cov_Inv *= hartlapfactor

Cov = np.diag(np.diag(Cov))
# Cov_Inv = np.linalg.inv(Cov)

plt.semilogy(np.diag(Cov))
hartlapfactor

### We compute the derivative of the bispectra with respect to $f_{\rm NL}$

In [ ]:
dBdf = (BispecP.mean(0)[:,-2]-BispecM.mean(0)[:,-2])/200

plt.semilogy(dBdf)
plt.show()

### Then the Fisher information is given by $$F = \sum_{TT'} \frac{\partial B_T}{f_{\rm NL}} C^{-1}_{TT'} \frac{\partial B_{T'}}{f_{\rm NL}}$$ and the measurement error is $\sigma_{f_{\rm NL}} = F^{-1/2}$

In [ ]:
FF = (dBdf.dot(Cov_Inv).dot(dBdf))
sigma = FF**-.5
sigma

### One can also estimate $f_{\rm NL}$ from the generated bispectra: $$\hat{f}_{\rm NL} = F^{-1}\sum{TT'}\frac{\partial B_T}{f_{\rm NL}}C^{-1}_{TT'} B_{T'}$$ where $B_{T'}$ is a measured bispectrum

In [ ]:
estimates_P = np.array([dBdf.dot(Cov_Inv).dot(BispecP[i,:,-2])/FF for i in tqdm(range(len(BispecP)))])
estimates_M = np.array([dBdf.dot(Cov_Inv).dot(BispecM[i,:,-2])/FF for i in tqdm(range(len(BispecM)))])
estimates_G = np.array([dBdf.dot(Cov_Inv).dot(BispecG[i,:,-2])/FF for i in tqdm(range(len(BispecG)))])

In [ ]:
estimates_P.mean(), estimates_M.mean(), estimates_G.mean()

In [ ]:
estimates_P.std(), estimates_M.std(), estimates_G.std()

In [ ]:
plt.hist(estimates_P,bins=100)
plt.hist(estimates_G,bins=100)
plt.hist(estimates_M,bins=100)
plt.show()

In [ ]:
plt.plot(BispecP.mean(0)[:,-2]*BispecP[0,:,:3].prod(1)*kF**3)
plt.plot(BispecM.mean(0)[:,-2]*BispecM[0,:,:3].prod(1)*kF**3)
plt.plot(BispecG.mean(0)[:,-2]*BispecG[0,:,:3].prod(1)*kF**3)

## Now we generate the corresponding maps

In [ ]:
np.random.seed(2)
fnls = np.random.uniform(fnl_range[0],fnl_range[1],num_sim).astype(np.float32)
fnls

In [ ]:
df_base = DensityField2D(BoxSize,grid, n_threads=1, cosmo_params=cosmo_params, verbose=True)

def make_map(fnl,seed, ls, qs, transfers, ells, d_A, cosmo, k_cut_low=None,k_cut_high=None,verbose=False):
    FFT_map = DensityField2D(BoxSize,grid, n_threads=1, d_A=d_A, ls=ls, qs=qs, transfers=transfers, ells=ells, cosmo_params=cosmo, verbose=verbose)
    FFT_map = DensityField2D(BoxSize,grid,n_threads=1)
    FFT_map.GenerateCAMBField(k_cut_low=k_cut_low,k_cut_high=k_cut_high,fnl=fnl,seed=seed)
    mapy = FFT_map.r_delta / FFT_map.r_delta.std()
    return mapy

## We generate maps with arbitrary $f_{\rm NL}$ to test parameter estimation using Unets

In [ ]:
from IPython.display import clear_output

n = num_sim

def save(dir, filename, map):
    with open(f'{dir}/{filename}', 'wb') as f:
        np.save(f, map)
        
print(f'base_name: {base_name} dir: {data_dir}\ndata file: {data_file}\nfnl file: {fnl_file}')

In [ ]:
%%time
maps = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(make_map)(fnl,None,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo,k_cut_high=0.25132741228718347) for fnl in fnls[:n]]))
save(data_dir, filename, maps)

In [ ]:
np.save(fnl_file,fnls)

In [ ]:
%%time
maps = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(make_map)(0.,seed,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo,k_cut_high=0.25132741228718347) for seed, fnl in enumerate(fnls)]))
np.save(data_dir, f"{base_name}_fnl0.npy", maps)

In [ ]:
%%time
maps = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(make_map)(100.,seed,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo, k_cut_high=0.25132741228718347) for seed, fnl in enumerate(fnls)]))
np.save(data_dir, f"{base_name}_fnl100.npy",maps)

## We generate non-Gaussian maps with k cuts to test large mode reconstruction using Unets

In [ ]:
%%time
mapsNG_highk = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(make_map)(fnl,seed,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo,k_cut_low=1.1*kF) for seed, fnl in enumerate(fnls)]))
np.save(data_dir,f"{base_name}_highkcut.npy",mapsNG_highk)
mapsNG_lowk = np.array(Parallel(n_jobs=-1,verbose=1)([delayed(make_map)(fnl,seed,df_base.Ls, df_base.qs, df_base.transfer, df_base.ells,df_base.d_A, df_base.cosmo, k_cut_high=1.1*kF) for seed, fnl in enumerate(fnls)]))
np.save(data_dir,f"{base_name}_lowkcut.npy",mapsNG_lowk)

In [ ]:
# import numpy as np
# import camb

# # Set up the cosmological parameters
# pars = camb.CAMBparams()
# pars.set_cosmology(H0=67.5, ombh2=0.022, omch2=0.122)
# pars.InitPower.set_params(ns=0.965, r=0)
# pars.set_for_lmax(2000)

# # Generate the power spectrum
# results = camb.get_results(pars)
# powers = results.get_cmb_power_spectra(pars, CMB_unit='muK')
# totCL = powers['total']

# # Generate the random Gaussian field
# np.random.seed(0)
# gaussian_field = np.random.normal(0, 1, (128, 128))

# # Apply the primordial power spectra
# k_vals = np.arange(1, 129)
# k_grid = np.meshgrid(k_vals, k_vals)
# k_norms = np.sqrt(k_grid[0] ** 2 + k_grid[1] ** 2)
# primordial_ps = np.interp(k_norms.flatten(), totCL[:, 0], totCL[:, 1])
# primordial_ps = primordial_ps.reshape(128, 128)

# # Apply non-Gaussianity
# f_NL = 1  # Set the non-Gaussianity parameter to 1
# gaussian_field = gaussian_field + f_NL * (gaussian_field ** 2 - 1)

# # Apply the CMB transfer functions
# ell_vals = np.arange(1, 2001)
# cmb_transfer = results.get_cmb_transfer_data(pars)
# transfer_func = np.interp(ell_vals, cmb_transfer.L, cmb_transfer.TT)
# transfer_func = transfer_func[:128]  # Truncate to match field size
# cmb_field = np.fft.fft2(gaussian_field * primordial_ps) * transfer_func
# cmb_field = np.real(np.fft.ifft2(cmb_field))

# # Display the result
# import matplotlib.pyplot as plt
# plt.imshow(cmb_field, cmap='jet', vmin=-100, vmax=100)
# plt.colorbar()
# plt.show()